# CellPLM Benchmarking (Mouse)

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 06.06.2026
- Mirrors the structure of `novae_benchmarking.ipynb` (mouse) but
  applies CellPLM instead of Novae, computing the same metrics so the
  two methods can be compared head-to-head.

- Clone CellPLM from GitHub into ```<root>/analysis/benchmarking```: ```git clone https://github.com/OmicsML/CellPLM.git```.
- Install the CellPLM conda environment ```../../envs/cellplm.yaml```
  (or the project's standard CellPLM env).
- **Species / gene mapping**: the CellPLM `20231027_85M` checkpoint was
  trained on a HUMAN single-cell corpus. Mouse genes are mapped to
  human orthologs BEFORE embedding via a 4-step pipeline (mygene
  mouse-DB -> Ensembl REST homology -> mygene human-DB direct -> NCBI
  HomoloGene fallback, and used as Ensembl IDs, since CellPLM's gene vocabulary is human ENSEMBL IDs, not HGNC symbols). Genes without an
  ortholog are reported and dropped explicitly (NOT silently ignored).
  All mapping logic is inlined in the notebook's function section (no
  external benchmarking-package dependency). Requires the `mygene`
  package + internet access (first call ~1-3 min for a few hundred
  genes).


## 1. Setup

### 1.1 Load Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../../utils")

In [ ]:
import math
import os
import warnings

import anndata as ad
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import sklearn.metrics as skm
import squidpy as sq
import torch
from CellPLM.pipeline.cell_embedding import CellEmbeddingPipeline
from CellPLM.utils import set_seed
from scib_metrics import ilisi_knn
from scib_metrics.nearest_neighbors import pynndescent
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import (
    adjusted_rand_score, normalized_mutual_info_score)
from sklearn.preprocessing import StandardScaler

from analysis_utils import *

### 1.2 Define Config

In [ ]:
method = 'CellPLM'

# General params -- matches novae_benchmarking.ipynb for the mouse dataset
dataset_name = 'mmb0-1b_smb1-1b_1p'
batches = ['harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes',
           'harmonised_starmap_plus_mouse_cns_batch15_shared_genes']
niche_key = 'tissue_region'
label_keys = [niche_key]
species = 'mouse'
n_neighs = 10
spot_sizes = {'MERFISH': 20,
              'STARmap PLUS': 100}
batch_key = 'batch'
n_pcs = None

# CellPLM params
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'
# Cross-species gene mapping: map mouse symbols to human orthologs
# BEFORE embedding (CellPLM 20231027_85M has a human gene vocab).
map_via_human_orthologs = True
# CellPLM's pretraining gene vocabulary is human ENSEMBL IDs (ENSG...),
# NOT HGNC symbols -- so map mouse genes to human Ensembl IDs.
gene_id_format = 'ensembl'   # 'ensembl' for CellPLM; 'symbol' otherwise
emb_key = 'cellplm_latent'

# Cluster params: CellPLM has no built-in domain assignment, so we
# pick a Leiden resolution that roughly matches novae's domain_level=30.
latent_leiden_resolution = 2.0
latent_cluster_key = f'leiden_res_{latent_leiden_resolution}'

### 1.3 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/nfs/team361/sb75/stjepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'
# CellPLM checkpoint directory (assumes the standard reproducibility
# layout from cellplm_benchmarking.ipynb).
cellplm_folder_path = (
    '/nfs/team361/sb75/terra-reproducibility/analysis/'
    'benchmarking/CellPLM')

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

pd.options.future.infer_string = False

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/{method}', exist_ok=True)

In [ ]:
set_seed(42)

### 1.5 Define Functions

In [ ]:
def make_large_cmap(n):
    base = plt.cm.get_cmap("turbo", n)   # or "hsv", "nipy_spectral"
    return [mcolors.to_hex(base(i)) for i in range(n)]

In [ ]:
def mmd_comparable(adata, emb_key, batch_key, n_sub=2000, n_sigma=1000):
    # Normalize to zero mean, unit variance
    X = StandardScaler().fit_transform(adata.obsm[emb_key])

    # Compute sigma on normalized data
    idx = np.random.choice(len(X), min(n_sigma, len(X)), replace=False)
    sub = X[idx]
    dists = np.linalg.norm(sub[:, None] - sub[None, :], axis=2)
    sigma = np.median(dists[dists > 0])

    # Split by batch
    batches = adata.obs[batch_key].unique()
    mask0 = adata.obs[batch_key] == batches[0]
    mask1 = adata.obs[batch_key] == batches[1]
    b0, b1 = X[mask0], X[mask1]

    idx0 = np.random.choice(len(b0), min(n_sub, len(b0)), replace=False)
    idx1 = np.random.choice(len(b1), min(n_sub, len(b1)), replace=False)

    def rbf(A, B):
        d = np.linalg.norm(A[:, None] - B[None, :], axis=2)
        return np.exp(-d**2 / (2 * sigma**2))

    b0s, b1s = b0[idx0], b1[idx1]
    return rbf(b0s, b0s).mean() + rbf(b1s, b1s).mean() - 2 * rbf(b0s, b1s).mean()

In [ ]:
def _combine_neighbor_graphs(adata: ad.AnnData,
                             n_neighs: int | None,
                             radius: float | None,
                             delaunay: bool,
                             include_self_loop: bool,
                             ) -> ad.AnnData:
    """Helper: combine neighbor graphs from knn / radius / delaunay."""
    if n_neighs is not None:
        sq.gr.spatial_neighbors(adata, coord_type='generic',
                                spatial_key='spatial', n_neighs=n_neighs,
                                set_diag=include_self_loop)
        knn_conn = adata.obsp['spatial_connectivities']
    if radius is not None:
        sq.gr.spatial_neighbors(adata, coord_type='generic',
                                spatial_key='spatial', radius=radius,
                                set_diag=include_self_loop)
        radius_conn = adata.obsp['spatial_connectivities']
        if n_neighs is not None:
            adata.obsp['spatial_connectivities'] = knn_conn.multiply(
                adata.obsp['spatial_connectivities'])
    if delaunay:
        sq.gr.spatial_neighbors(adata, coord_type='generic',
                                spatial_key='spatial', delaunay=True,
                                set_diag=include_self_loop)
        if n_neighs is not None:
            adata.obsp['spatial_connectivities'] = knn_conn.multiply(
                adata.obsp['spatial_connectivities'])
        if radius is not None:
            adata.obsp['spatial_connectivities'] = radius_conn.multiply(
                adata.obsp['spatial_connectivities'])
    return adata


def construct_neighbor_graph(adata: ad.AnnData,
                             n_neighs: int | None = None,
                             radius: float | None = None,
                             delaunay: bool = True,
                             include_self_loop: bool = False,
                             batch_key: str | None = None,
                             compute_neighbor_counts: bool = False,
                             ) -> ad.AnnData:
    """Compute spatial neighbor graph, optionally per-batch and aggregated.

    Same helper as the novae notebook so the spatial-graph step is
    identical across methods.
    """
    if batch_key is not None:
        first_idx = adata.obs.reset_index().groupby(batch_key).head(1).index
        batches = adata.obs.iloc[first_idx][batch_key].tolist()
        adata_batch_list = []
        for batch in batches:
            print(f"\nProcessing batch {batch}...")
            adata_batch = adata[adata.obs[batch_key] == batch]
            adata_batch = _combine_neighbor_graphs(
                adata_batch, n_neighs, radius, delaunay,
                include_self_loop)
            adata_batch_list.append(adata_batch)
        adata.obsp['spatial_connectivities'] = sp.block_diag(
            [a.obsp['spatial_connectivities'] for a in adata_batch_list],
            format="csr")
    else:
        adata = _combine_neighbor_graphs(
            adata, n_neighs, radius, delaunay, include_self_loop)
    if compute_neighbor_counts:
        adata.layers['X_neighborhood'] = (
            adata.obsp['spatial_connectivities'].T @ adata.X)
    return adata

In [ ]:
# --- Mouse -> human gene mapping helpers (SELF-CONTAINED) -----------
# Inlined verbatim from the shared ortholog pipeline
# (`_nicheformer_embedding.add_human_ortholog_ensembl_ids`), so this
# notebook has NO external dependency on the squint-reproducibility
# benchmarking package. CellPLM's 20231027_85M checkpoint has a HUMAN
# gene vocabulary, so mouse symbols are mapped to human orthologs
# BEFORE embedding (naive uppercasing silently drops every gene whose
# human ortholog has a different symbol). 4-step pipeline: mygene
# mouse-DB -> Ensembl REST homology -> mygene human-DB direct -> NCBI
# HomoloGene fallback, then ENSG -> HGNC symbol. Requires `mygene` +
# internet access.
import logging
from typing import Dict, Optional

logger = logging.getLogger(__name__)


def _ensembl_homology_lookup(
    mouse_ensembl_ids: list,
    verbose: bool = True,
    max_workers: int = 8,
    timeout: float = 15.0,
) -> Dict[str, str]:
    """
    Query the Ensembl REST homology endpoint for each mouse Ensembl
    gene ID and return a ``{mouse_id: human_id}`` dict.

    Uses ``ThreadPoolExecutor`` with a small number of concurrent
    requests; Ensembl REST allows ~15 req/s sustained.

    NOTE: For reproducible benchmarking against the published
    Nicheformer paper, prefer ``gene_mapper_path=<mart_export.txt>``
    (a static Biomart download) over this REST-based lookup.
    """
    try:
        import requests  # noqa: F401
    except Exception as e:
        raise ImportError(
            "_ensembl_homology_lookup requires `requests`. "
            f"Original error: {e!s}")
    import requests
    from concurrent.futures import ThreadPoolExecutor

    base = "https://rest.ensembl.org/homology/id/mus_musculus"
    headers = {"Accept": "application/json"}
    params_template = {
        "target_species": "homo_sapiens",
        "type": "orthologues",
        "format": "condensed",
    }

    def _query(mens_id: str) -> tuple:
        try:
            r = requests.get(
                f"{base}/{mens_id}",
                params=params_template,
                headers=headers,
                timeout=timeout,
            )
            if r.status_code != 200:
                return mens_id, None
            data = r.json()
            for entry in data.get("data", []):
                for h in entry.get("homologies", []):
                    target_id = h.get("id")
                    if target_id and str(target_id).startswith("ENSG"):
                        return mens_id, str(target_id)
            return mens_id, None
        except Exception:
            return mens_id, None

    out: Dict[str, str] = {}
    n = len(mouse_ensembl_ids)
    if verbose and n > 0:
        print(
            f"  ...calling Ensembl REST for {n} mouse Ensembl IDs "
            f"(this is the slow step; expect ~{max(5, n // 15)}s).",
            flush=True)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for mens_id, hens_id in ex.map(_query, mouse_ensembl_ids):
            if hens_id:
                out[str(mens_id)] = hens_id
    return out



def add_human_ortholog_ensembl_ids(
    adata: ad.AnnData,
    species: str = "mouse",
    gene_col_in: Optional[str] = None,
    gene_col_out: str = "human_ensembl_id",
    fallback_to_input: bool = True,
    verbose: bool = True,
    inplace: bool = True,
) -> ad.AnnData:
    """
    Populate ``adata.var[gene_col_out]`` with human Ensembl IDs derived
    from mouse gene symbols (or mouse Ensembl IDs).

    For each row in ``adata.var``:
      - Look up the gene symbol (or Ensembl ID) in mygene.info.
      - Find its mouse Ensembl ID.
      - Map mouse Ensembl -> human Ensembl via Ensembl REST homology.
      - If no mapping exists and ``fallback_to_input=True``, keep the
        original input string so it can be picked up by the outer-join
        with ``model.h5ad`` (mouse-only genes get dropped at the
        ``model.var_names`` subsetting step downstream).
    """
    if species.lower() not in ("mouse", "mus musculus"):
        # For non-mouse species (typically `human`), there's no ortholog
        # mapping to do — but we may still need to convert SYMBOLS to
        # Ensembl IDs so downstream code that expects ENSG in var
        # (Nicheformer, Geneformer) gets the right format. Decision tree:
        #
        #   - If the input column already holds Ensembl IDs (≥50% start
        #     with `ENSG`), just copy verbatim — nothing to do.
        #   - Otherwise treat the input as HGNC symbols and run a
        #     species-specific mygene lookup (e.g. CD3D -> ENSG00000167286
        #     when species=human).
        #
        # This makes the helper "smart" for human input too: every
        # downstream method that consumes `human_ensembl_id` gets a
        # proper Ensembl column regardless of whether the silver var_names
        # were symbols or already ENSG.
        if not inplace:
            adata = adata.copy()
        if gene_col_in is None:
            raw_ids = adata.var_names.astype(str).to_numpy()
        else:
            if gene_col_in not in adata.var.columns:
                raise KeyError(
                    f"gene_col_in={gene_col_in!r} not in adata.var.")
            raw_ids = adata.var[gene_col_in].astype(str).to_numpy()

        is_ensembl = np.array(
            [v.startswith("ENSG") for v in raw_ids], dtype=bool,
        )
        if is_ensembl.mean() > 0.5:
            if verbose:
                logger.info(
                    "add_human_ortholog_ensembl_ids: species=%r and "
                    "input looks like Ensembl IDs (%d/%d start with "
                    "ENSG); using verbatim.",
                    species, int(is_ensembl.sum()), len(raw_ids))
            adata.var[gene_col_out] = raw_ids
            return adata

        # Symbols → ENSG via mygene's species-specific DB.
        try:
            import mygene
        except Exception as e:
            raise ImportError(
                "add_human_ortholog_ensembl_ids requires the 'mygene' "
                "package for human-symbol → ENSG mapping. Install via "
                f"`pip install mygene`. Original error: {e!s}")
        if verbose:
            print(
                f"[add_human_ortholog_ensembl_ids] species={species!r}: "
                f"mapping {len(raw_ids)} symbols → Ensembl via mygene "
                f"({species}-DB lookup, no cross-species step).")
        mg = mygene.MyGeneInfo()
        # Strip species name for mygene (it accepts "human", "mouse", etc.).
        mg_species = species.lower().split()[0]
        df = mg.querymany(
            list(raw_ids),
            scopes=["symbol", "alias", "ensembl.gene"],
            species=mg_species,
            fields="ensembl.gene",
            returnall=False, as_dataframe=True,
        )
        df = df[~df.index.duplicated(keep="first")]

        def _first_ens(x):
            if isinstance(x, list) and x and isinstance(x[0], dict):
                return x[0].get("gene")
            if isinstance(x, dict):
                return x.get("gene")
            return x if isinstance(x, str) else None

        col = "ensembl.gene" if "ensembl.gene" in df.columns else "ensembl"
        sym_to_ens = df[col].apply(_first_ens) if col in df.columns else pd.Series(dtype=str)
        sym_to_ens = sym_to_ens[~sym_to_ens.index.duplicated(keep="first")]
        if fallback_to_input:
            mapped = np.array(
                [str(sym_to_ens.get(s, s)) for s in raw_ids], dtype=object,
            )
        else:
            mapped = np.array(
                [str(sym_to_ens.get(s, "")) for s in raw_ids], dtype=object,
            )
        adata.var[gene_col_out] = mapped
        n_mapped = int(np.sum(
            [str(v).startswith("ENSG") for v in mapped]
        ))
        if verbose:
            print(
                f"[add_human_ortholog_ensembl_ids] {n_mapped}/{len(raw_ids)} "
                f"symbols mapped to ENSG (species={species!r}).")
        return adata

    try:
        import mygene
    except Exception as e:
        raise ImportError(
            "add_human_ortholog_ensembl_ids requires the 'mygene' "
            "package. Install via `pip install mygene`. "
            f"Original error: {e!s}")

    if not inplace:
        adata = adata.copy()

    if gene_col_in is None:
        symbols = adata.var_names.astype(str).to_numpy()
    else:
        if gene_col_in not in adata.var.columns:
            raise KeyError(
                f"gene_col_in={gene_col_in!r} not in adata.var.")
        symbols = adata.var[gene_col_in].astype(str).to_numpy()

    if verbose:
        print(
            f"[add_human_ortholog_ensembl_ids] Mapping {len(symbols)} "
            "mouse gene IDs -> human Ensembl via mygene + Ensembl REST.")

    mg = mygene.MyGeneInfo()

    # ---- Step 1: input -> mouse Ensembl gene ID -------------------
    step1 = mg.querymany(
        symbols.tolist(),
        scopes=["symbol", "alias", "ensembl.gene"],
        species="mouse",
        fields="ensembl.gene",
        returnall=False, as_dataframe=True,
    )

    def _first_ens(x) -> Optional[str]:
        # Extract a single Ensembl gene ID from whatever shape mygene
        # returned for the `ensembl.gene` field. Supports BOTH human
        # (ENSG...) and mouse (ENSMUSG...) IDs because Step 1 queries
        # the mouse DB (returns ENSMUSG) and Step 3 queries the human
        # DB (returns ENSG) — using the same helper for both.
        if isinstance(x, list) and x and isinstance(x[0], dict):
            return x[0].get("gene")
        if isinstance(x, dict):
            return x.get("gene")
        if isinstance(x, str) and (
            x.startswith("ENSG") or x.startswith("ENSMUSG")
        ):
            return x
        return None

    if "ensembl.gene" in step1.columns:
        sym_to_mens = step1["ensembl.gene"].apply(_first_ens).dropna()
    elif "ensembl" in step1.columns:
        sym_to_mens = step1["ensembl"].apply(_first_ens).dropna()
    else:
        sym_to_mens = pd.Series(dtype=str)
    sym_to_mens = sym_to_mens[
        ~sym_to_mens.index.duplicated(keep="first")].astype(str)

    if verbose:
        print(
            f"  Step 1 (symbol -> mouse Ensembl):    "
            f"{len(sym_to_mens)}/{len(symbols)} mapped"
            + (f"   sample={list(sym_to_mens.items())[:3]}"
               if len(sym_to_mens) else ""))

    # ---- Step 2: mouse Ensembl -> human Ensembl via Ensembl REST --
    mens_to_hens: Dict[str, str] = {}
    if len(sym_to_mens):
        mens_to_hens = _ensembl_homology_lookup(
            list(sym_to_mens.unique()), verbose=verbose)
    if verbose:
        print(
            f"  Step 2 (mouse Ensembl -> human Ensembl via Ensembl "
            f"REST): {len(mens_to_hens)}/"
            f"{len(sym_to_mens.unique()) if len(sym_to_mens) else 0} "
            f"mapped"
            + (f"   sample={list(mens_to_hens.items())[:3]}"
               if mens_to_hens else ""))

    # ---- Compose Step 1+2: input symbol -> human Ensembl ----------
    sym_to_hens: Dict[str, str] = {}
    for sym, mens in sym_to_mens.items():
        h = mens_to_hens.get(mens)
        if h:
            sym_to_hens[str(sym)] = str(h)
    n_after_step2 = len(sym_to_hens)

    # ---- Step 3 (FALLBACK): direct mouse-symbol -> human Ensembl --
    # For genes Ensembl REST didn't cover, query mygene against the
    # HUMAN DB with the original (mouse-cased) symbol. Many mammalian
    # genes share their symbol spelling across species (modulo case),
    # and mygene's symbol/alias scopes are case-insensitive — so this
    # picks up the orthologs that have a directly-matching human gene
    # but for which Ensembl's curated homology table happens to be
    # incomplete. Cheap (one batched mygene call) and usually adds
    # 10-20 percentage points of coverage on mouse panels.
    unmapped_after_2 = [s for s in symbols if s not in sym_to_hens]
    if unmapped_after_2:
        try:
            step3 = mg.querymany(
                unmapped_after_2,
                scopes=["symbol", "alias"],
                species="human",
                fields="ensembl.gene",
                returnall=False, as_dataframe=True,
            )
            col = ("ensembl.gene" if "ensembl.gene" in step3.columns
                   else ("ensembl" if "ensembl" in step3.columns else None))
            if col is not None:
                step3_map = step3[col].apply(_first_ens).dropna()
                step3_map = step3_map[
                    step3_map.astype(str).str.startswith("ENSG")]
                step3_map = step3_map[
                    ~step3_map.index.duplicated(keep="first")]
                added = 0
                for sym, ensg in step3_map.items():
                    sym = str(sym)
                    if sym not in sym_to_hens:
                        sym_to_hens[sym] = str(ensg)
                        added += 1
                if verbose:
                    print(
                        f"  Step 3 (mouse symbol -> human Ensembl direct): "
                        f"+{added} new mappings ("
                        f"{n_after_step2}->{len(sym_to_hens)} total).")
        except Exception as exc:
            if verbose:
                print(f"  Step 3 (direct human-DB lookup) failed: {exc}")
    n_after_step3 = len(sym_to_hens)

    # ---- Step 4 (FALLBACK): NCBI HomoloGene via mygene -------------
    # Final fallback: for genes that even direct human-DB lookup
    # missed (mouse-specific symbols, or symbol-renamed orthologs),
    # query mygene's `homologene` field on the MOUSE Ensembl ID. The
    # `homologene.genes` field is a list of [taxid, gene_id] pairs;
    # taxid 9606 == human. We get back NCBI Entrez gene IDs, which we
    # then convert to Ensembl in a second mygene call.
    unmapped_after_3 = [s for s in symbols if s not in sym_to_hens]
    if unmapped_after_3 and len(sym_to_mens):
        try:
            mens_for_step4 = [
                str(sym_to_mens[s]) for s in unmapped_after_3
                if s in sym_to_mens.index
            ]
            if mens_for_step4:
                step4a = mg.querymany(
                    mens_for_step4,
                    scopes="ensembl.gene",
                    species="mouse",
                    fields="homologene",
                    returnall=False, as_dataframe=True,
                )
                # Extract human Entrez IDs (taxid 9606) from each row.
                human_entrez_per_mens: Dict[str, str] = {}
                if "homologene.genes" in step4a.columns:
                    for mens, val in step4a["homologene.genes"].items():
                        if not isinstance(val, list):
                            continue
                        for pair in val:
                            if (isinstance(pair, list) and len(pair) >= 2
                                    and int(pair[0]) == 9606):
                                human_entrez_per_mens[str(mens)] = str(pair[1])
                                break
                # Convert Entrez -> Ensembl in one batched call.
                if human_entrez_per_mens:
                    step4b = mg.querymany(
                        list(human_entrez_per_mens.values()),
                        scopes="entrezgene",
                        species="human",
                        fields="ensembl.gene",
                        returnall=False, as_dataframe=True,
                    )
                    col4 = ("ensembl.gene" if "ensembl.gene" in step4b.columns
                            else ("ensembl" if "ensembl" in step4b.columns
                                  else None))
                    if col4 is not None:
                        entrez_to_ensg = step4b[col4].apply(_first_ens).dropna()
                        entrez_to_ensg = entrez_to_ensg[
                            entrez_to_ensg.astype(str).str.startswith("ENSG")]
                        entrez_to_ensg = entrez_to_ensg[
                            ~entrez_to_ensg.index.duplicated(keep="first")]
                        entrez_to_ensg = entrez_to_ensg.astype(str).to_dict()
                        added = 0
                        for sym, mens in sym_to_mens.items():
                            sym = str(sym)
                            if sym in sym_to_hens:
                                continue
                            entrez = human_entrez_per_mens.get(str(mens))
                            if entrez and entrez in entrez_to_ensg:
                                sym_to_hens[sym] = entrez_to_ensg[entrez]
                                added += 1
                        if verbose:
                            print(
                                f"  Step 4 (NCBI HomoloGene via mygene): "
                                f"+{added} new mappings ("
                                f"{n_after_step3}->{len(sym_to_hens)} total).")
        except Exception as exc:
            if verbose:
                print(f"  Step 4 (HomoloGene fallback) failed: {exc}")

    # ---- Compose final ensembl_id column --------------------------
    n_mapped = len(sym_to_hens)
    out = []
    for s in symbols:
        h = sym_to_hens.get(s)
        if h is None:
            out.append(str(s) if fallback_to_input else "")
        else:
            out.append(str(h))
    adata.var[gene_col_out] = np.asarray(out, dtype=object)

    if verbose:
        print(
            f"[add_human_ortholog_ensembl_ids] FINAL: "
            f"{n_mapped}/{len(symbols)} input IDs mapped to human "
            "Ensembl. The rest "
            + ("fall back to the input string."
               if fallback_to_input else "are empty strings.")
            + f"\nSample of mapped pairs: "
            f"{list(sym_to_hens.items())[:5]}")
    return adata


def mouse_to_human(adata, species, target='ensembl'):
    """Convert a mouse-symbol AnnData to a human-gene AnnData.

    target='ensembl' : var_names = human Ensembl IDs  (CellPLM's vocab)
    target='symbol'  : var_names = human HGNC symbols (e.g. scGPT-spatial)

    Pipeline:
      1. mouse symbol -> human ENSG (4-step helper above).
      2. (only if target=='symbol') human ENSG -> human HGNC symbol.
    Genes without an ortholog are DROPPED (reported, not silently
    ignored). Genes mapping to the same human ID have their expression
    SUMMED. Returns a NEW AnnData with var_names = the chosen human IDs
    and var['feature_name'] = same; obs is preserved.
    """
    import mygene
    import scipy.sparse as _sp

    if target not in ('ensembl', 'symbol'):
        raise ValueError(f"target must be 'ensembl' or 'symbol', got {target!r}")

    # Step 1: mouse symbol -> human ENSG (always needed).
    print('  Step 1: mouse symbols -> human ENSG (mygene mouse + Ensembl '
          'REST + mygene human + NCBI HomoloGene)...')
    add_human_ortholog_ensembl_ids(
        adata, species=species, gene_col_in=None,
        gene_col_out='human_ensembl_id', fallback_to_input=False,
        verbose=True, inplace=True,
    )
    ensg = adata.var['human_ensembl_id']
    valid = ensg.notna() & ensg.astype(str).str.startswith('ENSG')
    print(f'    {int(valid.sum())}/{len(ensg)} mouse genes mapped to ENSG.')
    adata_h = adata[:, valid.to_numpy()].copy()

    if target == 'ensembl':
        # Use the human Ensembl IDs directly (CellPLM's gene vocab).
        final_ids = adata_h.var['human_ensembl_id'].astype(str).to_numpy()
    else:
        # Step 2: human ENSG -> human HGNC symbol.
        print('  Step 2: human ENSG -> human HGNC symbol (mygene)...')
        unique_ensg = adata_h.var['human_ensembl_id'].astype(str).unique().tolist()
        mg = mygene.MyGeneInfo()
        res = mg.querymany(
            unique_ensg, scopes='ensembl.gene', fields='symbol',
            species='human', as_dataframe=True,
        )
        res = res[~res.index.duplicated(keep='first')].dropna(subset=['symbol'])
        ensg_to_sym = res['symbol'].to_dict()
        mapped = adata_h.var['human_ensembl_id'].astype(str).map(ensg_to_sym)
        keep_mask = mapped.notna().to_numpy()
        print(f'    {int(keep_mask.sum())}/{len(mapped)} ENSG mapped to '
              'human HGNC symbol.')
        adata_h = adata_h[:, keep_mask].copy()
        final_ids = mapped[keep_mask].astype(str).to_numpy()

    # Sum expression across genes mapping to the same human id.
    unique_ids, inverse = np.unique(final_ids, return_inverse=True)
    n_dup = len(final_ids) - len(unique_ids)
    if n_dup > 0:
        print(f'    {n_dup} duplicate {target} ids after mapping; summing '
              'expression for duplicates.')
        agg = _sp.csr_matrix(
            (np.ones(len(inverse), dtype=np.float32),
             (np.arange(len(inverse)), inverse)),
            shape=(len(inverse), len(unique_ids)),
        )
        X = adata_h.X
        X_human = (X @ agg) if _sp.issparse(X) else (_sp.csr_matrix(X) @ agg)
        # X_human columns are in unique_ids (np.unique sorts).
        out_ids = unique_ids
    else:
        # No aggregation -> keep ORIGINAL column order; label with the
        # per-column ids (NOT the sorted unique_ids) to avoid mislabeling.
        X_human = adata_h.X
        out_ids = final_ids

    new_var = pd.DataFrame(
        {'feature_name': out_ids},
        index=pd.Index(out_ids, name='gene_name'),
    )
    out = ad.AnnData(
        X=X_human, obs=adata_h.obs.copy(), var=new_var,
        obsm=dict(adata_h.obsm),
    )
    print(f'  -> human AnnData (target={target}): n_obs={out.n_obs}, '
          f'n_vars={out.n_vars} (from original {adata.n_vars} mouse genes).')
    return out


def cellplm_gene_coverage(adata, pipeline):
    """Best-effort report of how many of adata's genes match CellPLM's
    pretraining gene vocabulary. CellPLM aligns genes internally (no
    vocab.json like scGPT), and the exact attribute holding the gene
    list varies across CellPLM versions, so this probes a few known
    paths and prints coverage if found -- purely informational, never
    fatal. Degenerate (all-zero) cells are handled downstream by the
    defensive finite-embedding mask.
    """
    model_genes = None
    for chain in [('model', 'gene_set'), ('fitted_genes',),
                  ('model', 'gene_list'), ('gene_list',),
                  ('model', 'genes')]:
        obj = pipeline
        try:
            for a in chain:
                obj = getattr(obj, a)
            if obj is not None and len(obj) > 0:
                model_genes = set(str(g) for g in obj)
                break
        except Exception:
            continue
    if model_genes is None:
        print('  (CellPLM gene vocab not introspectable via known '
              'attributes; CellPLM aligns genes internally. Skipping '
              'explicit coverage report.)')
        return
    genes = pd.Index(adata.var_names.astype(str))
    n_match = int(genes.isin(model_genes).sum())
    pct = 100.0 * n_match / max(len(genes), 1)
    print(f'  CellPLM vocab coverage: {n_match}/{len(genes)} '
          f'({pct:.1f}%) genes match the model gene set '
          f'(size={len(model_genes)}).')

## 2. Analysis

### 2.1 Retrieve Embeddings

In [ ]:
# Init CellPLM cell embedding pipeline
pipeline = CellEmbeddingPipeline(
    pretrain_prefix=PRETRAIN_VERSION,
    pretrain_directory=f'{cellplm_folder_path}/ckpt')
pipeline.model

In [ ]:
# Load adata batches and concatenate for inference. Matches the
# novae notebook's loading + niche-key + batch-key wiring exactly so
# downstream metrics are computed on the same cells.
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            print(file)

            if file == 'harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['ccf_region_name']
                adata.obs[batch_key] = file.replace('.h5ad', '')
            elif file == 'harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['Sub_molecular_tissue_region']
                adata.obs[batch_key] = file.replace('.h5ad', '')

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]
            print(adata.obs.columns)
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
print(adata)

In [ ]:
# Build the spatial neighbor graph (matches novae's pre-embedding step).
print('Computing spatial neighborhood...')
adata = construct_neighbor_graph(
    adata,
    n_neighs=n_neighs,
    batch_key=batch_key,
    radius=None,
    delaunay=None,
    include_self_loop=True,
    compute_neighbor_counts=False)

In [ ]:
# CellPLM preprocessing + cross-species gene mapping.
#   - `adata`           : original mouse-symbol data (keeps obs / region
#                          annotations / spatial coords for downstream).
#   - `adata_for_embed` : human-symbol data fed to CellPLM. Same n_obs
#                          as `adata` (only genes change), so the latent
#                          aligns 1:1 with `adata`.
#   - CellPLM reads gene names from adata.var_names and aligns them to
#     its (human) pretraining vocab internally; it expects raw /
#     normalized counts in .X and does its own preprocessing, so we
#     leave .X untouched.
if map_via_human_orthologs:
    print('=== Mapping mouse genes to human orthologs ===')
    adata_for_embed = mouse_to_human(
        adata.copy(), species=species, target=gene_id_format)
    assert adata_for_embed.n_obs == adata.n_obs, (
        'ortholog mapping changed n_obs; cannot align latent.')
else:
    adata_for_embed = adata.copy()
    adata_for_embed.var_names = adata_for_embed.var_names.astype(str)
    adata_for_embed.var_names_make_unique()

# Best-effort coverage report against CellPLM's gene vocab.
print('\n=== CellPLM gene-vocab coverage ===')
cellplm_gene_coverage(adata_for_embed, pipeline)

In [ ]:
# Generate zero-shot embeddings on the HUMAN-symbol AnnData, then
# attach the latent back to the original mouse-symbol `adata` (whose
# obs / region annotations / spatial coords we keep for downstream).
# `pipeline.predict` returns a tensor of shape (n_cells, embed_dim).
embedding = pipeline.predict(adata_for_embed, device=DEVICE)
latent = embedding.cpu().numpy().astype(np.float32)

# Alignment guard: `pipeline.predict` is a forward pass that returns
# one embedding row per input cell, in input order (no filtering). It
# returns a bare tensor (not an AnnData), so we can only check the
# row count -- verify it matches so a future CellPLM version that
# drops cells internally fails loud instead of silently misaligning.
if latent.shape[0] != adata.n_obs:
    raise RuntimeError(
        f'pipeline.predict returned {latent.shape[0]} rows but input '
        f'had {adata.n_obs} cells; cannot positionally align the '
        'latent. CellPLM may have filtered cells internally.')

# Defensive finite-mask: drop any cell whose embedding has NaN/Inf
# (e.g. an all-zero cell after vocab alignment). Subset BOTH the
# latent and `adata` so downstream metrics see only valid cells.
finite_mask = np.isfinite(latent).all(axis=1)
n_bad = int((~finite_mask).sum())
if n_bad > 0:
    print(f'  Dropping {n_bad}/{len(finite_mask)} cells with non-finite '
          'CellPLM embeddings.')
    adata = adata[finite_mask].copy()
    latent = latent[finite_mask]
assert np.isfinite(latent).all()

adata.obsm[emb_key] = latent
print(f"adata.obsm['{emb_key}'].shape = {adata.obsm[emb_key].shape}")
print(adata)

In [ ]:
# Write to disk
adata.write_h5ad(
    f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

### 2.2 Visualize UMAP

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                n_pcs=n_pcs, key_added=emb_key)
sc.tl.umap(adata, neighbors_key=emb_key)

n_pcs_string = f"_{n_pcs}_pcs" if n_pcs else ""
categories = adata.obs[batch_key].astype('category').cat.categories.tolist()
assert len(categories) == 2, "Expected exactly 2 categories for batch_key"
rename = dict(zip(sorted(categories), ["MERFISH", "STARmap PLUS"]))
adata.obs[batch_key] = adata.obs[batch_key].map(rename).astype("category")

sc.pl.umap(
    adata,
    color=batch_key,
    title="Technology",
    legend_loc="right margin",
    legend_fontsize=10,
    show=False,
)
for collection in plt.gca().collections:
    collection.set_rasterized(True)
plt.savefig(
    f"{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_{emb_key}_umap_by_{batch_key}{n_pcs_string}.svg",
    format="svg", bbox_inches="tight"
)
plt.show()
plt.close()

for label_key in label_keys:
    n_cats = adata.obs[label_key].nunique()
    palette = make_large_cmap(n_cats)
    sc.pl.umap(adata, color=label_key, palette=palette, show=False)
    for collection in plt.gca().collections:
        collection.set_rasterized(True)
    plt.savefig(
        f"{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_{emb_key}_umap_by_{label_key}{n_pcs_string}.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
# Write to disk
adata.write_h5ad(
    f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

### 2.3 Visualize Niches

In [ ]:
adata = sc.read_h5ad(
    f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                n_pcs=n_pcs, key_added=emb_key)
sc.tl.leiden(adata,
             neighbors_key=emb_key,
             key_added=latent_cluster_key,
             resolution=latent_leiden_resolution,
             flavor="igraph",
             n_iterations=2)
print(f"#clusters @ res={latent_leiden_resolution}: "
      f"{adata.obs[latent_cluster_key].nunique()}")

In [ ]:
# Write to disk
adata.write_h5ad(
    f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded_clustered.h5ad')

In [ ]:
# Spatial plots, one per sample (technology). Mirrors the novae loop.
n_clusters = adata.obs[latent_cluster_key].nunique()
cluster_palette = make_large_cmap(n_clusters)
mapped_palette = dict(zip(
    sorted(adata.obs[latent_cluster_key].unique(), key=lambda x: int(x)),
    cluster_palette,
))

samples = adata.obs[batch_key].unique().tolist()
for sample in samples:
    adata_sample = adata[adata.obs[batch_key] == sample]
    sc.pl.spatial(
        adata_sample,
        color=latent_cluster_key,
        palette=mapped_palette,
        spot_size=spot_sizes[sample],
        title=sample,
        legend_loc="right margin",
        legend_fontsize=10,
        show=False,
    )
    fig = plt.gcf()
    for ax in fig.axes:
        for child in ax.get_children():
            if isinstance(child, mpl.image.AxesImage):
                child.set_rasterized(True)
        for collection in ax.collections:
            collection.set_rasterized(True)
    plt.savefig(
        f"{benchmarking_artifacts_folder_path}/{method}/tissue_{sample}_by_{latent_cluster_key}.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

### 2.4 Compute Metrics

In [ ]:
# (Re-)load the clustered AnnData so the metrics section is
# self-contained. Harmony batch correction has been removed -- all
# metrics below are computed directly on the raw CellPLM embedding
# `emb_key`.
adata = sc.read_h5ad(
    f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded_clustered.h5ad')

# Average cosine similarity of the (raw) embeddings -- direct
# replacement for the same step in the novae notebook.
X = adata.obsm[emb_key]
X = X / np.linalg.norm(X, axis=1, keepdims=True)
n = X.shape[0]
S = X.sum(axis=0)
avg_cos_sim = (S @ S - n) / (n * (n - 1))
print(f'{emb_key} average cosine similarity: {avg_cos_sim}')

cos_sim_result = pd.DataFrame([{
    "emb_key": emb_key,
    "avg_cosine_similarity": avg_cos_sim,
}])

out_path = (
    f"{benchmarking_artifacts_folder_path}/{method}/"
    f"{dataset_name}_{emb_key}_avg_cosine_similarity.csv"
)
cos_sim_result.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")

In [ ]:
# NMI / ARI vs ground-truth tissue regions (one number per region key,
# computed on the cells that have a label for that key).
results = []
for region_key in ["Sub_molecular_tissue_region", "ccf_region_name"]:
    mask = adata.obs[region_key].notna()
    labels_true = adata.obs.loc[mask, region_key]
    labels_pred = adata.obs.loc[mask, latent_cluster_key]
    nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
    ari = skm.adjusted_rand_score(labels_true, labels_pred)
    print(f"NMI ({region_key}): {nmi:.4f}")
    print(f"ARI ({region_key}): {ari:.4f}")
    results.append({"region_key": region_key, "NMI": nmi, "ARI": ari})

df = pd.DataFrame(results)
df.to_csv(
    f"{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_{latent_cluster_key}_niche_identification_metrics.csv",
    index=False
)

In [ ]:
# Batch integration metrics on the raw embedding.
results = []

# iLISI
batch = adata.obs[batch_key].values
knn = pynndescent(adata.obsm[emb_key], n_neighbors=90)
ilisi = float(np.mean(ilisi_knn(knn, batch)))
print(f"iLISI: {ilisi:.4f}")
results.append({"metric": "iLISI", "score": ilisi})

# ASW
asw = silhouette_score(
    adata.obsm[emb_key],
    adata.obs[batch_key],
    metric="euclidean",
    sample_size=10000
)
print(f"ASW batch: {asw:.4f}")
results.append({"metric": "ASW", "score": asw})

# MMD
mmd_score = mmd_comparable(adata, emb_key, batch_key)
print(f"MMD: {mmd_score:.4f}")
results.append({"metric": "MMD", "score": mmd_score})

df = pd.DataFrame(results)
df.to_csv(
    f"{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_{emb_key}_batch_integration_metrics.csv",
    index=False
)

In [ ]:
# Resolution sweep: NMI / ARI as a function of Leiden resolution on the
# raw embedding -- mirrors the novae notebook's sweep.
N_STEPS   = 4      # points above and below the base resolution
STEP_SIZE = 0.1
MIN_RES   = 0.1

region_keys = ["Sub_molecular_tissue_region", "ccf_region_name"]

base_int = round(latent_leiden_resolution / STEP_SIZE)
resolutions = [
    round(step * STEP_SIZE, 1)
    for step in range(base_int - N_STEPS, base_int + N_STEPS + 1)
    if step * STEP_SIZE >= MIN_RES
]

all_results = []

for res in resolutions:
    cluster_key = f"{emb_key}_clusters_res_{res}"
    try:
        import rapids_singlecell as rsc  # noqa: F401
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=cluster_key,
                      resolution=res,
                      flavor="igraph",
                      n_iterations=2)
    except (ImportError, NameError):
        sc.tl.leiden(adata,
                     neighbors_key=emb_key,
                     key_added=cluster_key,
                     resolution=res,
                     flavor="igraph",
                     n_iterations=2)

    n_clusters = adata.obs[cluster_key].nunique()
    print(f"\n-- Resolution {res:.1f}  ({n_clusters} clusters) --")

    for region_key in region_keys:
        mask        = adata.obs[region_key].notna()
        labels_true = adata.obs.loc[mask, region_key]
        labels_pred = adata.obs.loc[mask, cluster_key]
        nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
        ari = skm.adjusted_rand_score(labels_true, labels_pred)
        print(f"  {region_key:40s}  NMI={nmi:.4f}  ARI={ari:.4f}")
        all_results.append({
            "resolution": res,
            "n_clusters": n_clusters,
            "region_key": region_key,
            "NMI":        nmi,
            "ARI":        ari,
        })

df = pd.DataFrame(all_results)
df = df[["resolution", "n_clusters", "region_key", "NMI", "ARI"]]
out_path = (
    f"{benchmarking_artifacts_folder_path}/{method}/"
    f"{dataset_name}_{emb_key}_niche_identification_metrics_res_sweep.csv"
)
df.to_csv(out_path, index=False)
print(f"\nSaved {len(df)} rows -> {out_path}")